# 10 — SVM con kernel RBF

Un modello per notebook, dall'input alla lettura degli errori. Usiamo soltanto
KDDTrain+, tutte le 41 feature e le cinque famiglie Normal, DoS, Probe, R2L e U2R.
Ripetiamo il confronto sugli stessi cinque split stratificati 80/20, seed 1–5.
Questo è un confronto di sviluppo; il test ufficiale resta separato.

**Percorso:** lettura → split → preprocessing → training → predizione → metriche
→ confronto con i risultati precedenti. Nessuna ricerca di iperparametri.

## Come ragiona questo modello

`SVC`, da `sklearn.svm`, cerca separazioni con un margine ampio. Il kernel RBF
permette confini non lineari, dando più somiglianza ai punti vicini nello spazio
delle feature. Nella classificazione multiclasse SVC addestra confronti fra coppie.

Configurazione iniziale fissata prima delle prove: `C=1.0`, `gamma='scale'`,
`class_weight=None`. C regola il compromesso fra margine ed errori sul training;
gamma regola quanto localmente agisce il kernel. `gamma='scale'` viene calcolato
dai dati trasformati del solo training. `probability=False` evita la stima
aggiuntiva di probabilità; `max_iter=-1` non impone un taglio alle iterazioni.
La cache del kernel è 512 MB. Nessuna griglia di tuning.

Il primo fit (seed1) serve anche a misurare i tempi e resta una delle cinque prove.
Non riduciamo il dataset e non lo ripetiamo per completare il confronto.
[Documentazione SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html).

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import sys
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

project_root = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(project_root))
from src.data import COLUMN_NAMES, FEATURE_COLUMNS, CATEGORICAL_COLUMNS
from src.targets import ATTACK_TO_FAMILY, FAMILY_ORDER

## 1. Leggere i dati e distinguere input e target

`pd.read_csv` legge il CSV locale senza intestazione. Schema e mapping sono
costanti del progetto, non funzioni degli autori del dataset. Le 41 feature
escludono `attack_label` e `difficulty`; il target è la famiglia dell'attacco.

In [ ]:
train_path = project_root / 'data/raw/KDDTrain+.csv'
string_types = {column: 'string' for column in (*CATEGORICAL_COLUMNS, 'attack_label')}
data = pd.read_csv(train_path, header=None, names=COLUMN_NAMES, dtype=string_types)
unknown = data['attack_label'].isna() | ~data['attack_label'].isin(ATTACK_TO_FAMILY)
if unknown.any():
    raise ValueError('Etichette mancanti o non mappate: riesaminare il target.')

X = data.loc[:, list(FEATURE_COLUMNS)].copy()
y = data['attack_label'].map(ATTACK_TO_FAMILY).rename('family')
assert X.shape == (125973, 41) and X.index.equals(pd.RangeIndex(len(X)))
assert not X.isna().any().any()
display(X.head())
display(y.value_counts().reindex(FAMILY_ORDER).rename('record'))

## 2. Fissare i cinque split prima del training

`train_test_split` è la funzione originale sklearn. `stratify=y` conserva circa
le proporzioni delle classi. I seed diversi cambiano i record separati fra train
e validation; un seed fissato rende riproducibile ciascuna prova. Questi modelli,
con questa configurazione, non richiedono casualità aggiuntiva nel training.

Carichiamo il riferimento congelato del notebook05: i cinque split devono
coincidere anche nell'ordine dei record e nelle risposte, non soltanto nelle
dimensioni. Il riferimento si rigenera con05 dai risultati04, senza riaddestrare.

In [ ]:
from sklearn.model_selection import train_test_split

reference_path = project_root / 'artifacts/rare_class_reference'
reference = json.loads((reference_path / 'results.json').read_text(encoding='utf-8'))
source_sha256 = hashlib.sha256(train_path.read_bytes()).hexdigest()
assert source_sha256 == reference['source_sha256']
assert reference['seeds'] == [1, 2, 3, 4, 5]
assert reference['validation_size'] == 0.2 and reference['feature_count'] == 41
assert reference['family_order'] == list(FAMILY_ORDER)
with np.load(reference_path / 'predictions.npz', allow_pickle=False) as saved:
    reference_arrays = {key: saved[key].copy() for key in saved.files}

seeds = [1, 2, 3, 4, 5]
prepared_splits = []
split_records = []
for run, seed in enumerate(seeds, start=1):
    X_train, X_validation, y_train, y_validation = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed,
    )
    train_indices = X_train.index.to_numpy()
    validation_indices = X_validation.index.to_numpy()
    assert not np.intersect1d(train_indices, validation_indices).size
    prepared_splits.append({
        'run': run, 'seed': seed, 'train_indices': train_indices,
        'validation_indices': validation_indices, 'y_validation': y_validation,
    })
    split_records.append({
        'run': run, 'seed': seed, 'train_rows': len(X_train), 'validation_rows': len(X_validation),
        'train_indices_sha256': hashlib.sha256(train_indices.tobytes()).hexdigest(),
        'validation_indices_sha256': hashlib.sha256(validation_indices.tobytes()).hexdigest(),
    })
assert split_records == reference['splits']
display(pd.DataFrame(split_records)[['run', 'seed', 'train_rows', 'validation_rows']])

In [ ]:
for split in prepared_splits:
    run = split['run']
    np.testing.assert_array_equal(
        split['validation_indices'], reference_arrays[f'run_{run:02d}_validation_indices'],
    )
    np.testing.assert_array_equal(
        np.asarray(split['y_validation']), reference_arrays[f'run_{run:02d}_truth'],
    )
print('Tutti i record e i target coincidono: possiamo procedere al training.')

## 3. Definire il preprocessing e la Pipeline

- Tre categoriche (`protocol_type`, `service`, `flag`): `OneHotEncoder`, con
  categorie nuove in validation ignorate e rappresentazione densa.
- Le altre 38 colonne: `StandardScaler`, che sottrae la media e divide per la
  deviazione standard del training. Nessuna feature viene eliminata.

Per SVM la scala influenza le distanze del kernel. Per GaussianNB standardizziamo
anche per evitare che le unità delle colonne più grandi dominino la stabilizzazione
globale `var_smoothing`: è una nostra scelta dichiarata. Le dummy restano 0/1.

La Pipeline apprende encoder e scaler **durante `fit(X_train, y_train)`**.
Alla validation applica soltanto `transform`, attraverso `predict`. `clone` crea
una Pipeline nuova per ogni split: i parametri appresi non passano fra prove.
[Documentazione StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html).

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

numeric_columns = [column for column in FEATURE_COLUMNS if column not in CATEGORICAL_COLUMNS]
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), list(CATEGORICAL_COLUMNS)),
    ('numeric', StandardScaler(), numeric_columns),
], remainder='drop')
classifier = SVC(
        C=1.0, kernel='rbf', gamma='scale', class_weight=None,
        probability=False, tol=1e-3, max_iter=-1, cache_size=512,
    )
model_template = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', classifier),
])
model_id = 'svm_rbf'
display(model_template)

## 4. Training: cinque fit, soltanto sui record di training

Questo è il passaggio che apprende il modello. Qui non calcoliamo ancora le
predizioni di validation. Stampiamo il tempo di ogni fit e conserviamo i cinque
modelli in memoria per il passaggio successivo.

In [ ]:
fitted_models = {}
fit_seconds_by_seed = {}
parameters_by_seed = {}
for split in prepared_splits:
    seed = split['seed']
    train_indices = split['train_indices']
    X_train = X.loc[train_indices]
    y_train = y.loc[train_indices]
    model = clone(model_template)
    print(f"Inizio training {model_id}, seed={seed} ({len(X_train)} record).", flush=True)
    start = perf_counter()
    model.fit(X_train, y_train)
    fit_seconds_by_seed[str(seed)] = perf_counter() - start
    fitted_models[seed] = model
    parameters_by_seed[str(seed)] = model.named_steps['classifier'].get_params()
    print(f"Fit completato: {fit_seconds_by_seed[str(seed)]:.2f} secondi.", flush=True)

## 5. Controllare che il preprocessing abbia visto soltanto il training

Confrontiamo vocabolario e statistiche apprese con i record effettivi di training.
Registriamo anche eventuali categorie presenti soltanto in validation. Questi
controlli non cambiano dati, iperparametri o modello dopo aver osservato i risultati.

In [ ]:
preprocessing_checks = []
classifier_checks = []
for split in prepared_splits:
    seed = split['seed']
    X_train = X.loc[split['train_indices']]
    X_validation = X.loc[split['validation_indices']]
    y_train = y.loc[split['train_indices']]
    model = fitted_models[seed]
    fitted_preprocessor = model.named_steps['preprocessor']
    encoder = fitted_preprocessor.named_transformers_['categorical']
    scaler = fitted_preprocessor.named_transformers_['numeric']
    np.testing.assert_allclose(scaler.mean_, X_train[numeric_columns].mean(), rtol=1e-10, atol=1e-10)
    np.testing.assert_allclose(scaler.var_, X_train[numeric_columns].var(ddof=0), rtol=1e-10, atol=1e-10)
    assert int(scaler.n_samples_seen_) == len(X_train)
    categories = {}
    unseen = {}
    for column, learned in zip(CATEGORICAL_COLUMNS, encoder.categories_):
        assert set(learned) == set(X_train[column].unique())
        categories[column] = learned.tolist()
        unseen[column] = sorted(set(X_validation[column]) - set(learned))
    preprocessing_checks.append({
        'run': split['run'], 'seed': seed, 'numeric_columns': numeric_columns,
        'scaler_mean': scaler.mean_.tolist(), 'scaler_var': scaler.var_.tolist(),
        'scaler_n_samples_seen': int(scaler.n_samples_seen_),
        'categories': categories, 'unseen_validation': unseen,
    })
    classifier = model.named_steps['classifier']

    assert classifier.fit_status_ == 0, 'SVM non convergente: riportare il problema, non usare il risultato in silenzio.'
    details = {'classes': classifier.classes_.tolist(), 'fit_status': int(classifier.fit_status_),
               'n_support': classifier.n_support_.tolist(), 'n_iter': classifier.n_iter_.tolist(),
               'gamma_effective': float(classifier._gamma)}
    classifier_checks.append({'seed': seed, **details})
display(pd.DataFrame(classifier_checks))
print('Encoder, scaler e classificatore verificati sui soli record di training.')

## 6. Predizione: applicare il modello alla validation

`predict` usa i parametri già appresi: qui non c'è un altro `fit`. Salviamo
posizioni originali, target e predizioni di ogni run per ricontrollare le metriche.

In [ ]:
prediction_arrays = {}
predict_seconds_by_seed = {}
for split in prepared_splits:
    run, seed = split['run'], split['seed']
    validation_indices = split['validation_indices']
    X_validation = X.loc[validation_indices]
    start = perf_counter()
    y_pred = fitted_models[seed].predict(X_validation)
    predict_seconds_by_seed[str(seed)] = perf_counter() - start
    prediction_arrays[f'run_{run:02d}_validation_indices'] = validation_indices.copy()
    prediction_arrays[f'run_{run:02d}_truth'] = np.asarray(split['y_validation'], dtype=str)
    prediction_arrays[f'run_{run:02d}_predicted'] = np.asarray(y_pred, dtype=str)
    print(f"Predizione seed={seed}: {predict_seconds_by_seed[str(seed)]:.2f} secondi.", flush=True)

## 7. Metriche per run e per famiglia

La matrice ha **righe = classe vera, colonne = classe predetta**. Normalizzando
ciascuna riga, la diagonale è il recall: la quota dei casi reali riconosciuti.
`quota_errori = 1 - recall`. Per capire quanto fidarsi di una previsione di attacco
leggiamo anche precision e falsi positivi.

Se una classe non viene mai predetta, impostiamo precision/F1 a zero con
`zero_division=0`, mantenendo supporto ed errori visibili. Le metriche si calcolano
prima per run, poi si mediano; non ricaviamo la precision dalla matrice media.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

run_rows = []
class_rows = []
count_matrices = []
normalized_matrices = []
for run, seed in enumerate(seeds, start=1):
    truth = prediction_arrays[f'run_{run:02d}_truth']
    predicted = prediction_arrays[f'run_{run:02d}_predicted']
    report = classification_report(truth, predicted, labels=FAMILY_ORDER, output_dict=True, zero_division=0)
    counts = confusion_matrix(truth, predicted, labels=FAMILY_ORDER)
    count_matrices.append(counts)
    normalized_matrices.append(confusion_matrix(truth, predicted, labels=FAMILY_ORDER, normalize='true'))
    run_rows.append({
        'run': run, 'seed': seed, 'accuracy': accuracy_score(truth, predicted),
        'macro_f1': f1_score(truth, predicted, labels=FAMILY_ORDER, average='macro', zero_division=0),
        'weighted_f1': f1_score(truth, predicted, labels=FAMILY_ORDER, average='weighted', zero_division=0),
    })
    for i, family in enumerate(FAMILY_ORDER):
        support, correct = int(counts[i].sum()), int(counts[i, i])
        class_rows.append({
            'run': run, 'seed': seed, 'famiglia': family,
            'support': support, 'corretti': correct, 'errori': support - correct,
            'falsi_positivi': int(counts[:, i].sum() - correct),
            'quota_errori': (support - correct) / support if support else np.nan,
            'precision': report[family]['precision'], 'recall': report[family]['recall'],
            'f1': report[family]['f1-score'],
        })
run_results = pd.DataFrame(run_rows)
class_results = pd.DataFrame(class_rows)

In [ ]:
overall_summary = run_results[['accuracy', 'macro_f1', 'weighted_f1']].agg(['mean', 'std', 'min', 'max'])
class_summary = class_results.groupby('famiglia', sort=False)[
    ['precision', 'recall', 'f1', 'quota_errori', 'falsi_positivi']
].agg(['mean', 'std', 'min', 'max'])
mean_matrix = np.stack(normalized_matrices).mean(axis=0)
std_matrix = np.stack(normalized_matrices).std(axis=0, ddof=1)
display(run_results.round(6))
display(overall_summary.round(6))
display(class_summary.reindex(FAMILY_ORDER).round(4))
display(class_results.loc[class_results['famiglia'].isin(['R2L', 'U2R'])].round(4))

## 8. Visualizzare la confusion matrix normalizzata

È la **media delle cinque matrici normalizzate per riga**, con scala comune 0–1.
La tabella successiva mostra la deviazione standard campionaria delle celle.
La matrice descrive errori fra classi, non correlazioni fra feature.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

figure, axis = plt.subplots(figsize=(6.6, 5.4), layout='constrained')
ConfusionMatrixDisplay(mean_matrix, display_labels=FAMILY_ORDER).plot(
    ax=axis, cmap='Blues', values_format='.2f', colorbar=True, im_kw={'vmin': 0, 'vmax': 1},
)
axis.set(title=f'{model_id}: media su 5 split', xlabel='Classe predetta', ylabel='Classe vera')
plt.show()
display(pd.DataFrame(std_matrix, index=FAMILY_ORDER, columns=FAMILY_ORDER).round(4))

## 9. Salvare risultati e provenienza

Gli artefatti di questo modello sono separati dai precedenti: CSV leggibili,
JSON con configurazione e controlli, NPZ Unicode rileggibile con `allow_pickle=False`.
Non salviamo i modelli addestrati. I tempi si riferiscono a questa macchina.
Il nuovo schema non usa `max_depth`, che non è un parametro di questi modelli.

In [ ]:
artifacts = project_root / 'artifacts' / model_id
artifacts.mkdir(parents=True, exist_ok=True)
run_results.to_csv(artifacts / 'run_metrics.csv', index=False)
class_results.to_csv(artifacts / 'class_metrics.csv', index=False)
overall_summary.to_csv(artifacts / 'overall_summary.csv')
class_summary.to_csv(artifacts / 'class_summary.csv')
np.savez_compressed(artifacts / 'predictions.npz', **prediction_arrays)
figure.savefig(artifacts / 'mean_confusion_matrix.png', dpi=150)
pd.DataFrame(mean_matrix, index=FAMILY_ORDER, columns=FAMILY_ORDER).to_csv(artifacts / 'confusion_mean.csv')
pd.DataFrame(std_matrix, index=FAMILY_ORDER, columns=FAMILY_ORDER).to_csv(artifacts / 'confusion_std.csv')
experiment = {
    'schema_version': 2, 'model_id': model_id,
    'evaluation_set': 'repeated validation from KDDTrain+ only',
    'source': reference['source'], 'source_sha256': source_sha256,
    'rows': len(X), 'feature_count': X.shape[1], 'family_order': list(FAMILY_ORDER),
    'validation_size': 0.2, 'stratified': True, 'seeds': seeds, 'std_ddof': 1,
    'randomness_varied': ['train_validation_split'], 'splits': split_records,
    'classifier_parameters_by_seed': parameters_by_seed,
    'preprocessing': {'categorical': 'OneHotEncoder(handle_unknown=ignore, sparse_output=False)',
                      'numeric': 'StandardScaler()', 'fit_scope': 'training partition only'},
    'preprocessing_checks': preprocessing_checks, 'classifier_checks': classifier_checks,
    'fit_seconds_by_seed': fit_seconds_by_seed, 'predict_seconds_by_seed': predict_seconds_by_seed,
    'run_metrics': run_results.to_dict(orient='records'),
    'class_metrics': class_results.to_dict(orient='records'),
    'normalized_confusion_mean': mean_matrix.tolist(),
    'normalized_confusion_std': std_matrix.tolist(),
    'confusion_counts': [matrix.tolist() for matrix in count_matrices],
    'unique_validation_counts': reference['unique_validation_counts'],
    'versions': {'python': platform.python_version(), **{
        package: importlib.metadata.version(package) for package in ['numpy', 'pandas', 'scikit-learn', 'matplotlib']
    }},
    'limits': [
        'Overlapping repeated holdouts; measurements are not independent.',
        'Rare U2R cases are reused; repetitions do not create new observations.',
        'Known duplicate-feature overlap is retained; see DATA_AUDIT.md.',
        'One initial configuration, all 41 features; no official-test estimate or exact paper replication.',
        'Models use different preprocessing/class-weight choices explicitly documented; no isolated causal attribution.',
    ],
}
(artifacts / 'results.json').write_text(json.dumps(experiment, indent=2), encoding='utf-8')
timings = pd.DataFrame({
    'seed': seeds, 'fit_seconds': [fit_seconds_by_seed[str(seed)] for seed in seeds],
    'predict_seconds': [predict_seconds_by_seed[str(seed)] for seed in seeds],
})
timings.to_csv(artifacts / 'timings.csv', index=False)
display(timings.round(3))

## 10. Confrontare gli stessi casi con i modelli precedenti

Riutilizziamo le predizioni salvate, senza nuovi fit degli alberi. Ricontrolliamo
fonte, seed, indici e target prima di affiancare i risultati; le statistiche vengono
ricalcolate dalle predizioni. Il primo albero è senza pesi, albero pesato e RF sono
`balanced`, i nuovi NB/SVM senza pesi aggiunti: confrontiamo queste configurazioni,
non attribuiamo ogni differenza al solo algoritmo. Il preprocessing è dichiarato
per ciascun metodo e appreso dal rispettivo training.

In [ ]:
saved_models = {
    'decision_tree': 'rare_class_reference',
    'decision_tree_balanced': 'decision_tree_balanced',
    'random_forest_balanced': 'random_forest_balanced',
    'gaussian_naive_bayes': 'gaussian_naive_bayes',
}
comparison_overall_rows = []
comparison_class_rows = []
comparison_matrices = {}
for compared_id, folder in {**saved_models, model_id: model_id}.items():
    path = project_root / 'artifacts' / folder
    metadata = json.loads((path / 'results.json').read_text(encoding='utf-8'))
    if metadata.get('schema_version', 1) != (2 if compared_id in {'gaussian_naive_bayes', 'svm_rbf'} else 1):
        raise ValueError(f'Formato artefatto non supportato per {compared_id}.')
    if metadata.get('model_id', 'decision_tree' if compared_id == 'decision_tree' else None) != compared_id:
        raise ValueError(f'Identità artefatto diversa dal modello atteso: {compared_id}.')
    assert metadata['source_sha256'] == source_sha256 and metadata['seeds'] == seeds
    assert metadata['feature_count'] == 41 and metadata['family_order'] == list(FAMILY_ORDER)
    assert metadata['splits'] == split_records
    matrices = []
    with np.load(path / 'predictions.npz', allow_pickle=False) as saved:
        for run, seed in enumerate(seeds, start=1):
            np.testing.assert_array_equal(saved[f'run_{run:02d}_validation_indices'],
                                          prediction_arrays[f'run_{run:02d}_validation_indices'])
            truth = saved[f'run_{run:02d}_truth']
            np.testing.assert_array_equal(truth, prediction_arrays[f'run_{run:02d}_truth'])
            suffix = 'predicted' if metadata.get('schema_version') == 2 else 'depth_15'
            predicted = saved[f'run_{run:02d}_{suffix}']
            counts = confusion_matrix(truth, predicted, labels=FAMILY_ORDER)
            matrices.append(confusion_matrix(truth, predicted, labels=FAMILY_ORDER, normalize='true'))
            report = classification_report(truth, predicted, labels=FAMILY_ORDER, output_dict=True, zero_division=0)
            comparison_overall_rows.append({
                'model': compared_id, 'seed': seed, 'accuracy': accuracy_score(truth, predicted),
                'macro_f1': report['macro avg']['f1-score'], 'weighted_f1': report['weighted avg']['f1-score'],
            })
            for i, family in enumerate(FAMILY_ORDER):
                comparison_class_rows.append({
                    'model': compared_id, 'seed': seed, 'famiglia': family,
                    'precision': report[family]['precision'], 'recall': report[family]['recall'],
                    'f1': report[family]['f1-score'], 'quota_errori': 1 - report[family]['recall'],
                    'falsi_positivi': int(counts[:, i].sum() - counts[i, i]),
                })
    comparison_matrices[compared_id] = np.stack(matrices).mean(axis=0)
comparison_overall = pd.DataFrame(comparison_overall_rows)
comparison_classes = pd.DataFrame(comparison_class_rows)
comparison_summary = comparison_overall.groupby('model', sort=False)[
    ['accuracy', 'macro_f1', 'weighted_f1']
].agg(['mean', 'std', 'min', 'max'])
comparison_class_summary = comparison_classes.groupby(['model', 'famiglia'], sort=False)[
    ['precision', 'recall', 'f1', 'quota_errori', 'falsi_positivi']
].agg(['mean', 'std', 'min', 'max'])
display(comparison_summary.round(6))
display(comparison_class_summary.loc[(slice(None), ['R2L', 'U2R']), :].round(4))
comparison_overall.to_csv(artifacts / 'comparison_run_metrics.csv', index=False)
comparison_classes.to_csv(artifacts / 'comparison_class_metrics.csv', index=False)
comparison_summary.to_csv(artifacts / 'comparison_overall_summary.csv')
comparison_class_summary.to_csv(artifacts / 'comparison_class_summary.csv')

In [ ]:
metrics = ['precision', 'recall', 'f1', 'quota_errori', 'falsi_positivi']
reference_metrics = comparison_classes.loc[comparison_classes['model'].eq('decision_tree')]
paired = comparison_classes.loc[comparison_classes['model'].eq(model_id)].merge(
    reference_metrics, on=['seed', 'famiglia'], suffixes=('_model', '_reference'), validate='one_to_one',
)
paired_deltas = paired[['seed', 'famiglia']].copy()
for metric in metrics:
    paired_deltas[metric] = paired[f'{metric}_model'] - paired[f'{metric}_reference']
paired_deltas.to_csv(artifacts / 'paired_deltas_vs_tree.csv', index=False)
display(paired_deltas.groupby('famiglia', sort=False)[metrics].agg(['mean', 'std', 'min', 'max']).round(4))

In [ ]:
comparison_figure, axes = plt.subplots(2, 3, figsize=(15, 9), layout='constrained')
for axis, (compared_id, matrix) in zip(axes.flat, comparison_matrices.items()):
    ConfusionMatrixDisplay(matrix, display_labels=FAMILY_ORDER).plot(
        ax=axis, cmap='Blues', values_format='.2f', colorbar=False, im_kw={'vmin': 0, 'vmax': 1},
    )
    axis.set(title=compared_id, xlabel='Classe predetta', ylabel='Classe vera')
for axis in list(axes.flat)[len(comparison_matrices):]:
    axis.set_visible(False)
comparison_figure.colorbar(axes.flat[0].images[0], ax=list(axes.flat)[:len(comparison_matrices)], shrink=0.7)
comparison_figure.suptitle('Stessi cinque split: medie delle matrici normalizzate per classe vera')
comparison_figure.savefig(artifacts / 'comparison_confusion_matrices.png', dpi=150)
plt.show()

## 11. Come leggere questi risultati

- Guardare la diagonale della matrice normalizzata per identificare le classi
  perse, poi precision e falsi positivi per valutare gli allarmi generati.
- Controllare tutte le classi; su R2L e U2R leggere anche i singoli seed e il supporto.
- Media e DS campionaria descrivono queste cinque prove: le validation si
  sovrappongono e includono soltanto 31 U2R distinti, non cinque nuovi dataset.
- Accuracy e weighted F1 possono nascondere errori sulle minoranze. Nessun
  vincitore automatico e nessuna equivalenza con il test ufficiale del paper.

I risultati misurati e il controllo indipendente sono riportati nella sezione finale.

## SVM RBF — esito misurato e ricontrollato

Cinque fit, nessuna ricerca di iperparametri. **Macro F1 medio 0.840586
± 0.035241 DS**, accuracy media 99,32%.

| Famiglia | Precision media | Recall medio | Errore medio | FP medi/run |
|---|---:|---:|---:|---:|
| U2R | 62,33% | 26,00% | 74,00% | 1.6 |
| R2L | 88,45% | 84,52% | 15,48% | 22.0 |

Per leggere il compromesso, la RF già disponibile aveva precision/recall U2R
78,34%/60,00%; per R2L
94,83%/97,09%.
In questa configurazione SVM ha precision e recall inferiori alla RF per entrambe
le famiglie rare: perde in media il 74% degli U2R e il 15,48% degli R2L. Migliora
molto il confronto complessivo rispetto a Naive Bayes, ma non migliora il risultato
RF già disponibile. Le matrici affiancate sono nel notebook10. Non scegliamo
automaticamente dal solo macro F1 e non affermiamo che una
configurazione iniziale esaurisca le possibilità del metodo.

Tutti e cinque i fit SVM sono convergenti (`fit_status_=0`); tempo medio fit 29.20s,
predizione 8.04s su questa macchina. Il seed1 è incluso, non ripetuto.

Metriche ricalcolate indipendentemente dai NPZ; stessi indici/target degli altri
modelli. Le cinque prove riutilizzano 31 U2R distinti: non sono campioni indipendenti.


## 12. Confronto diretto con Random Forest

Stessi cinque split80/20 seed1–5, stessi record e target, tutte41feature.
Risultati ricalcolati dalle predizioni salvate, nessun nuovo addestramento.

| Modello | Accuracy media | Macro F1 media ± DS | Weighted F1 medio |
|---|---:|---:|---:|
| Gaussian Naive Bayes | 74,18% | 0.4344 ± 0.0383 | 0.7834 |
| SVM RBF | 99,32% | 0.8406 ± 0.0352 | 0.9931 |
| Random Forest balanced | 99,83% | 0.9200 ± 0.0312 | 0.9982 |

| Modello | Precision U2R | Recall U2R | Errore U2R | FP U2R/run | Precision R2L | Recall R2L | Errore R2L |
|---|---:|---:|---:|---:|---:|---:|---:|
| Gaussian Naive Bayes | 0,80% | 88,00% | 12,00% | 1089.0 | 6,06% | 48,94% | 51,06% |
| SVM RBF | 62,33% | 26,00% | 74,00% | 1.6 | 88,45% | 84,52% | 15,48% |
| Random Forest balanced | 78,34% | 60,00% | 40,00% | 2.2 | 94,83% | 97,09% | 2,91% |

| Classe vera | Naive Bayes: errore | SVM: errore | RF: errore |
|---|---:|---:|---:|
| Normal | 28,38% | 0,67% | 0,18% |
| DoS | 8,13% | 0,06% | 0,03% |
| Probe | 78,65% | 1,60% | 0,29% |
| R2L | 51,06% | 15,48% | 2,91% |
| U2R | 12,00% | 74,00% | 40,00% |


**Interpretazione:** la RF balanced è il compromesso migliore fra queste tre
configurazioni nelle prove correnti. Su R2L ha sia precision sia recall maggiori;
su U2R migliora entrambi rispetto a SVM. SVM produce 1,6 FP U2R/run contro2,2 RF,
ma riconosce soltanto il26% degli U2R contro60%: il calo degli allarmi accompagna
molti più attacchi persi. NB riconosce88%U2R, ma con precision0,80% e1089FP/run;
non è un miglioramento utilizzabile in questo confronto. RF conserva comunque
un erroreU2R40%, ancora elevato. Anche la classe Probe distingue nettamente NB.

Le righe delle matrici sono le classi vere, le colonne quelle predette; l'errore
è1−diagonale. I falsi positivi non si leggono dal solo recall: affiancare precision.
Sono configurazioni iniziali diverse: RF usa pesi balanced e numeriche non scalate;
NB/SVM nessun peso aggiunto e numeriche standardizzate. Non attribuire causalmente
ogni differenza al solo algoritmo. Le validation si sovrappongono,31U2R distinti,
nessun test ufficiale o replica numerica del paper.


![Confusion matrix dei tre modelli](../docs/assets/nb_svm_rf_comparison/confusion_matrices.png)